# PCA of foci metrics: the multivariate picture

This notebook looks at **all foci metrics at once**. Each cell is described by about a dozen numbers: how many foci, how big, how round, how enriched, how clustered, and so on.
**Principal component analysis (PCA)** compresses them into a few new axes, the *principal components* (PCs), that capture most of the variation between cells. It answers:

* **Which metrics vary together?** For example, cells with many foci may also have small foci.
* **What are the main "directions" of variation?** For example, PC1 might be "amount of phase separation" and PC2 "droplet shape".
* **Do the conditions occupy different regions of this space**, and is the difference consistent between experiments?

### How it relates to `METRICS.ipynb`

| | `METRICS.ipynb` | `PCA_v2.ipynb` |
|---|---|---|
| Question | *Which metric changes, and by how much?* | *Is the overall foci phenotype different, and along which combination of metrics?* |
| Output | one test and effect size per metric | a 2D map of cells, an overall test, a distance from the control |
| Role | **main result** | overview, and a summary of many small changes that add up |

A typical result: *"no single metric differs strongly, but together the mutant is shifted along PC1, which combines foci number, area fraction and signal in foci."*

### Workflow

| Step | Section | What it does |
|---|---|---|
| 1 | Parameters | files, features, transformations |
| 2 | Design check | replicates and cells per condition |
| 3 | Feature preparation | missing values, log transform, standardisation |
| 4 | Correlations | which features are redundant |
| 5 | PCA fit | how many components are meaningful |
| 6 | Interpreting the components | what each PC measures |
| 7 | Conditions in PCA space | scatter, confidence ellipses, replicate centroids |
| 8 | Statistics | PERMANOVA, distances from the reference, PC scores per replicate |
| 9 | Expression check | whether the main PC is just expression level |
| 10 | Outliers | unusual cells to check in the images |
| 11 | Save and report | tables, figures, reporting checklist |

### The replicate is still the unit of statistics

As in `METRICS.ipynb`, cells from one experiment are not independent. A PCA plot with 500 cells per condition can look very convincing even when the difference is a single dish effect.
This notebook therefore shows **replicate centroids** (diamonds) on the PCA map, computes confidence ellipses that **include the variability between replicates**, and reports **replicate-level tests** next to the cell-level ones. Cell-level tests are descriptive only.

In [ ]:
%matplotlib inline
from PCA_v2_utils import *

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# 1. Parameters

### `data`: one entry per condition × biological replicate

This is the same format as in `METRICS.ipynb`, so you can copy the list from there.
* `condition`: the label on the plots. The order of first appearance sets the order and the colours.
* `replicate`: the replicate ID within the condition. Use the same IDs for replicates done together.
* `path`: the `aggregated_data.csv` of that condition and replicate.

All files must come from the **same segmentation method and parameters**.

### `EXCLUDE`: cells to leave out

These are cells you have checked in the images and decided to remove, e.g. a segmentation failure or a cut-off nucleus. Section 10 prints ready-to-paste entries for flagged outliers. Each entry is `(condition, replicate, image_id)`.
Leave it empty at first.

In [ ]:
# Root directory where the data is stored
root_dir = "/mnt/c/Users/helen/Desktop/data"

# One entry per condition x replicate. Template for 3 replicates:
#   {"condition": "WT", "replicate": "1", "path": f"{root_dir}/WT/rep1/gmm/aggregated_data.csv"},
#   {"condition": "WT", "replicate": "2", "path": f"{root_dir}/WT/rep2/gmm/aggregated_data.csv"},
#   {"condition": "WT", "replicate": "3", "path": f"{root_dir}/WT/rep3/gmm/aggregated_data.csv"},
data = [
    {"condition": "WT",   "replicate": "1", "path": f"{root_dir}/WT/gmm/aggregated_data.csv"},
    {"condition": "MGS3", "replicate": "1", "path": f"{root_dir}/MGS3/gmm/aggregated_data.csv"},
]

COMPARISONS = "vs_reference"   # "vs_reference" or "all_pairs"
REFERENCE = "WT"               # baseline condition
EXCLUDE = []                   # cells to leave out: (condition, replicate, image_id)

N_PERM = 9999                  # permutations for PERMANOVA
N_BOOT = 2000                  # bootstrap iterations for ellipses and distance CIs
CONFIDENCE = 0.95              # confidence level of ellipses and CIs
ALPHA = 0.05                   # significance level for the q-values
SAVE_RESULTS = True
output_dir = Path(root_dir) / "pca_v2"   # tables and figures are saved here

### Features

These are the metrics that enter the PCA, grouped as in `METRICS.ipynb`. Two principles:

1. **Include foci properties, not expression or imaging.** `mean_intensity`, `max_intensity` and `min_intensity` (focus brightness) and `nucleus_mean_intensity` (expression) depend on how much protein the cell makes and on laser and gain settings.
   If included, PC1 often becomes "bright vs. dim cells". Expression is used in section 9 instead, to *check* the PCs.
2. **Avoid near-duplicates.** PCA treats every feature as one vote. `area`, `equivalent_diameter_area` and `perimeter` all measure size, so together they would give size triple weight. Section 4 lists redundant pairs, so you can come back and comment out one feature of each pair.

### Transformations

* **`LOG_FEATURES`** are replaced by log₁₀(1 + x). Counts, sizes and ratios are right-skewed: a few cells with 300 foci or huge foci would dominate the PCA. The log makes the distributions closer to symmetric and the PCA more stable.
* **Standardisation:** after that, every feature is z-scored (mean 0, SD 1), so features in px, ratios and fractions are comparable. This happens automatically.

### Missing values

PCA needs a value for **every feature in every cell**, but some metrics are missing by design. Cells without foci have no per-focus metrics, and cells with fewer than `MIN_FOCI` foci have no Gini or Clark–Evans ratio.
* `MISSING = "drop"` removes such cells. It's clean, but the PCA then describes **only cells with enough foci**. Section 3 shows how many cells are kept per condition; if a condition loses many more cells than the others, that is itself a result, so report it.
* `MISSING = "median"` fills gaps with the overall median and keeps all cells, but it pulls those cells artificially towards the centre of the plot.

If many cells are dropped, remove the spacing metrics (`size_gini`, `clark_evans_ratio`, `nn_gap`, `fraction_touching`) rather than filling them.

In [ ]:
feature_groups = {
    "Amount of phase separation": [
        "n_foci",
        "foci_density",
        "area_fraction",
        "fraction_signal_in_foci",
        "partition_coefficient",
    ],
    "Focus properties": [
        "area",
        "eccentricity",
        "solidity",
        "size_gini",
    ],
    "Spatial organisation": [
        "clark_evans_ratio",
        "nn_gap",
        "fraction_touching",
    ],
}
features = [f for group in feature_groups.values() for f in group]

# log10(1 + x) before standardisation (right-skewed features, all >= 0)
LOG_FEATURES = ["n_foci", "foci_density", "area_fraction", "partition_coefficient", "area", "nn_gap"]

MISSING = "drop"               # "drop" or "median"

# Cells with fewer foci than this get no value for the metric (as in METRICS.ipynb)
MIN_FOCI = {
    "size_gini": 4,
    "clark_evans_ratio": 4,
    "nn_gap": 2,
    "fraction_touching": 2,
}

REDUNDANCY_THRESHOLD = 0.85    # |Spearman rho| above which two features are flagged as redundant

# 2. Load the data and check the design

The table shows the number of cells per condition and replicate.
* With **≥ 2 replicates** per condition, the replicate-level tests in section 8 are computed.
* With **one replicate**, only cell-level tests are possible. They show whether the cells differ, not whether the result would reproduce.

In [ ]:
df = load_conditions(data)
if EXCLUDE:
    key = list(zip(df["condition"].astype(str), df["replicate"].astype(str), df["image_id"].astype(str)))
    drop = [k in set(map(tuple, EXCLUDE)) for k in key]
    print(f"Excluded {sum(drop)} cells listed in EXCLUDE")
    df = df[~np.array(drop)]
df = apply_min_foci(df, MIN_FOCI)
TEST_LEVEL = check_design(df)
palette = condition_palette(list(df["condition"].cat.categories))
design_table(df)

# 3. Feature preparation

### Cells kept

`kept_%` is the share of each condition's cells that have all features (with `MISSING = "drop"`). If it differs a lot between conditions, the PCA compares different sub-populations. The usual cause is that one condition has many cells with zero or very few foci.
In that case, remove the spacing metrics from `features`, or report the share of cells with foci separately (it's in `METRICS.ipynb`).

In [ ]:
print("Fraction of cells with no value, per feature and condition:")
display(missing_table(df, features))

data_pca, Z, kept = prepare_features(df, features, log_features=LOG_FEATURES, missing=MISSING)
features_used = list(Z.columns)
kept

### Distributions after transformation and standardisation

Every feature should now be roughly bell-shaped and centred at 0.
* A feature that is still **strongly skewed** (a long tail to one side) is worth adding to `LOG_FEATURES`.
* A feature with **only a few distinct values**, e.g. `n_foci` when most cells have 1–3 foci, carries little information and can make the PCA look striped.

In [ ]:
fig = plot_feature_distributions(Z)
plt.show()

# 4. Correlations between features

This is the Spearman correlation between every pair of features over all cells. Features are ordered by clustering, so correlated features sit next to each other.

* **Strongly correlated pairs** (|ρ| ≥ `REDUNDANCY_THRESHOLD`) are listed below. They measure nearly the same thing and give it double weight in the PCA. Comment out one of each pair in `features` and rerun.
  `foci_density` and `n_foci` are a typical pair, as are `area_fraction` and `fraction_signal_in_foci`.
* **Moderate correlations** (0.4–0.8) are what PCA is for: it combines them into one component.
* A feature **uncorrelated with all others** will get a PC of its own, or end up in a minor PC.

In [ ]:
corr = correlation_matrix(Z)
fig = plot_correlation(corr)
if SAVE_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "feature_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

redundant = redundant_pairs(corr, REDUNDANCY_THRESHOLD)
print(f"Redundant pairs (|rho| >= {REDUNDANCY_THRESHOLD}):" if len(redundant) else "No redundant pairs.")
redundant

# 5. PCA: how many components matter?

**Bars** show the share of the total variation captured by each component, and the **black line** the cumulative share.
The **red dashed line** is the *chance level* from **parallel analysis**: each feature is shuffled independently, which destroys every correlation, and the PCA is refitted 200 times.
A component is **meaningful** if its bar is above the red line (dark blue bars). Components below it describe noise, and you shouldn't interpret them.

Usually only 2–3 components are meaningful. If **none** is above chance, the features are mostly independent, and per-metric analysis (`METRICS.ipynb`) is more informative than PCA.

In [ ]:
pca, scores = fit_pca(Z, data_pca)
chance = parallel_analysis(Z)
above = pca.explained_variance_ratio_ * 100 > chance
n_meaningful = int(np.argmax(~above)) if (~above).any() else len(above)   # leading PCs above chance
N_PCS = max(2, min(n_meaningful, 4))   # PCs used below (at least 2 for plotting)
print(f"Meaningful components (above chance): {n_meaningful}. Using PC1-PC{N_PCS} below.")

fig = plot_scree(pca, chance)
if SAVE_RESULTS:
    fig.savefig(output_dir / "scree.png", dpi=300, bbox_inches="tight")
plt.show()

# 6. What does each component measure?

### Feature–component correlations

Each cell of the heatmap is the **correlation between a feature and a PC**, from −1 to 1.
To name a PC, look at the features with the largest absolute values in its column:

* features with the **same sign** increase together along that PC;
* features with **opposite signs** trade off: when one goes up, the other goes down;
* values below ~0.3 in absolute terms contribute little.

For example, if PC1 has +0.9 for *Foci per cell*, *Area fraction* and *Signal in foci*, it means **"amount of phase separation"**: cells to the right of the PCA map have more condensed protein.
**The sign of a PC is arbitrary.** Running the same analysis on new data may flip it, so always interpret it through the features.

In [ ]:
loadings = loadings_table(pca, features_used, n_pcs=N_PCS)
fig = plot_loadings(loadings, pca)
if SAVE_RESULTS:
    fig.savefig(output_dir / "loadings.png", dpi=300, bbox_inches="tight")
plt.show()

### Correlation circle

This is the same information for PC1 and PC2 as arrows:
* an **arrow's direction** shows where cells with high values of that feature lie on the PCA map;
* an **arrow's length** shows how well PC1 and PC2 represent the feature (1 = perfectly, on the circle). Short, faded arrows are features that mostly vary along other PCs;
* the **angle between arrows** reflects their correlation: same direction means positive, opposite means negative, right angle means unrelated.

Keep this figure next to the PCA map in section 7. It translates "the mutant lies to the upper right" into "the mutant has more and rounder foci".

In [ ]:
fig = plot_correlation_circle(loadings, pca)
if SAVE_RESULTS:
    fig.savefig(output_dir / "correlation_circle.png", dpi=300, bbox_inches="tight")
plt.show()

# 7. Conditions in PCA space

* **Small dots:** individual cells.
* **Large dots:** condition centroids (mean position).
* **Ellipses:** the 95 % confidence region of each **centroid**, not of the cells. They come from a hierarchical bootstrap (resampling replicates, then cells), so they **include the variability between experiments**. Non-overlapping ellipses suggest different centroids.
* **Diamonds** (with ≥ 2 replicates): replicate centroids. If the diamonds of two conditions separate cleanly, the difference reproduces across experiments. If they intermingle, it doesn't, however clear the cell clouds look.

The cell clouds of different conditions usually **overlap strongly**: cell-to-cell variability is large. What matters is whether the *centroids* are shifted, and in which direction. Read that direction with the correlation circle above.

In [ ]:
fig = plot_pca_scatter(scores, pca, palette=palette, confidence=CONFIDENCE, n_boot=N_BOOT)
if SAVE_RESULTS:
    fig.savefig(output_dir / "pca_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

### One panel per condition

Here each condition is shown in colour, with all other cells in grey. This makes it easier to see the **shape** of each cloud: whether a condition shifts as a whole, gains a sub-population (a separate cluster), or just spreads more.

In [ ]:
fig = plot_pca_facets(scores, pca, palette=palette)
if SAVE_RESULTS:
    fig.savefig(output_dir / "pca_facets.png", dpi=300, bbox_inches="tight")
plt.show()

if N_PCS >= 3:
    fig = plot_pca_scatter(scores, pca, pcs=("PC1", "PC3"), palette=palette,
                           confidence=CONFIDENCE, n_boot=N_BOOT, title="PC1 vs PC3")
    plt.show()

# 8. Statistics

### 8a. PERMANOVA: do the conditions differ in their overall foci phenotype?

PERMANOVA tests whether the condition **centroids** differ in the **full space of all features** (z-scored, Euclidean), not just in PC1–PC2. The *p*-value comes from shuffling the condition labels.

| Column | Meaning |
|---|---|
| `level = cells` | every cell is an observation. **Descriptive only**, because cells of one experiment are not independent |
| `level = replicates` | replicate centroids are the observations (*n* = number of replicates). **Base conclusions on this one** |
| `R2` | fraction of the total variation explained by condition. It's an effect size: 0.05 means 5 %. At cell level it is usually small, because cell-to-cell variability is large; at replicate level it is larger |
| `p`, `q` | permutation *p*-value, and the Benjamini–Hochberg q-value over the pairwise comparisons |
| `p_min` | **the smallest p this design can produce** at replicate level. With 3 vs. 3 replicates there are only 20 ways to split them, so *p* can't go below 0.1, however different the conditions are. Pairwise replicate-level PERMANOVA needs **≥ 4 replicates per condition** to reach *p* < 0.05. With 3, rely on section 8c |
| `p_permdisp` | PERMDISP: do the conditions differ in **spread**? If it's significant, a significant PERMANOVA may reflect different variability rather than (only) a shifted centroid. Check the facets above |

In [ ]:
pairs = comparison_pairs(scores["condition"].cat.categories, reference=REFERENCE, mode=COMPARISONS)
mv = multivariate_tests(scores, Z, pairs, n_perm=N_PERM)
mv.round(4)

### 8b. How far is each condition from the reference?

This is the distance between each condition's centroid and the reference centroid, in the **full z-scored feature space**.
It's measured in **standard deviations** (SD units): the square root of the summed squared standardised differences over all features.
The bar is a 95 % hierarchical-bootstrap CI, which includes replicate variability. Stars come from the PERMANOVA q-value: replicate level when available, otherwise cell level.

`main_features` in the table lists the three features with the largest standardised differences, with their sign (+ = higher than the reference). For each one, `METRICS.ipynb` gives its test and effect size.

A distance whose CI starts well above 0 suggests a different phenotype. The bootstrap distance is never exactly 0, though: even two identical conditions give a small positive distance. Use the PERMANOVA result for the yes/no question, and the distance for "how much".

In [ ]:
dist = centroid_distances(scores, Z, REFERENCE, n_boot=N_BOOT, confidence=CONFIDENCE)
fig = plot_centroid_distances(dist, REFERENCE, palette, mv=mv)
if SAVE_RESULTS:
    fig.savefig(output_dir / "centroid_distances.png", dpi=300, bbox_inches="tight")
plt.show()
dist.round(3)

### 8c. PC scores per condition (SuperPlots)

The same replicate-aware approach as `METRICS.ipynb` is applied to the **PC scores**. Each meaningful PC is treated as a new metric, e.g. "amount of phase separation" = PC1.
The test is Welch's *t*-test on **replicate medians**, which works with 3 replicates, unlike pairwise replicate-level PERMANOVA. The cell-level test is used when there's only one replicate.

This tells you **along which component** a condition differs. Combine it with section 6 to describe the difference in biological terms.

In [ ]:
pc_cols = [f"PC{i + 1}" for i in range(N_PCS)]
pc_names = {pc: f"{pc} ({pca.explained_variance_ratio_[i] * 100:.0f}%)" for i, pc in enumerate(pc_cols)}
pc_res = compare_metrics(scores, pc_cols, pairs, n_boot=200)
q_col = "q_replicates" if TEST_LEVEL == "replicates" else "q_cells"

fig = plot_superplots(scores, pc_cols, labels={pc: "score" for pc in pc_cols}, names=pc_names,
                      results=pc_res, q_col=q_col, title="PC scores per condition")
if SAVE_RESULTS:
    fig.savefig(output_dir / "pc_superplots.png", dpi=300, bbox_inches="tight")
plt.show()

# Ratios are meaningless for PC scores (centred at 0): use the difference and Cliff's delta
pc_res["difference"] = pc_res["median"] - pc_res["median_baseline"]
pc_res[["metric", "condition", "baseline", "n_rep", "n_cells", "median", "median_baseline",
        "difference", "cliffs_delta", "p_replicates", "q_replicates", "p_cells", "q_cells"]].round(4)

# 9. Expression-level check

If PC1 is strongly correlated with **expression level** (`nucleus_mean_intensity`), the main axis of variation is "how much protein the cell has". A condition shift along PC1 may then just mean the cells express more (see section 7 of `METRICS.ipynb`).

* **Left:** cells coloured by expression. A colour gradient along a PC means that PC follows expression.
* **Table:** Spearman ρ between each PC and expression, over all cells and within each condition. |ρ| > 0.5 is strong.

In [ ]:
EXPR = "nucleus_mean_intensity"
if EXPR in scores.columns:
    fig = plot_pca_colored(scores, pca, EXPR)
    plt.show()
    rows = {}
    for pc in pc_cols:
        row = {"all cells": stats.spearmanr(scores[pc], scores[EXPR]).statistic}
        for cond, g in scores.groupby("condition", observed=True):
            row[str(cond)] = stats.spearmanr(g[pc], g[EXPR]).statistic if len(g) >= 5 else np.nan
        rows[pc] = row
    display(pd.DataFrame(rows).T.round(2))
else:
    print(f"{EXPR} not in the data: rerun agg.py with the current version.")

# 10. Outliers

**Multivariate outliers** are cells whose *combination* of features is unusual for their condition, even if no single value is extreme, e.g. very many foci that are also very large.
They're detected with the **Mahalanobis distance** of each cell from **its own condition's centroid** in the full feature space, at a strict 99.9 % cut-off.
Detection is done *within* each condition. Pooling all cells would flag a whole condition whose phenotype really differs, and removing those cells would erase the effect you're looking for.

**Don't remove outliers automatically.** Open their images (`image_id` below) and their `panels/` overlays:
* if the cell is a segmentation failure (a cut-off nucleus, two nuclei, a mitotic cell, a staining artefact), add it to `EXCLUDE` in the Parameters and rerun;
* if the cell is genuine but unusual, **keep it**. Rare phenotypes are biology, and removing them biases the result.

Report how many cells were excluded and why.

In [ ]:
flags = find_outliers(scores, Z)
fig = plot_outliers(scores, pca, flags, palette=palette)
plt.show()

out = scores.loc[outlier_order(flags), ["condition", "replicate", "image_id", "n_foci"]].join(flags["mahalanobis"])
out.insert(0, "#", range(1, len(out) + 1))   # numbers shown on the plot
print(f"{len(out)} of {len(scores)} cells flagged. Per condition:")
display(out.groupby("condition", observed=True).size().rename("outliers").to_frame())
print("To exclude after checking the images, paste into EXCLUDE:")
print(",\n".join(f'    ("{r.condition}", "{r.replicate}", "{r.image_id}")' for r in out.itertuples()))
out.round(2)

# 11. Save and report

In [ ]:
if SAVE_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output_dir / "pca_scores.csv", index=False)
    loadings.to_csv(output_dir / "loadings.csv")
    mv.to_csv(output_dir / "permanova.csv", index=False)
    dist.to_csv(output_dir / "centroid_distances.csv", index=False)
    pc_res.to_csv(output_dir / "pc_score_tests.csv", index=False)
    out.to_csv(output_dir / "outliers.csv", index=False)
    pd.DataFrame({"explained_variance_%": pca.explained_variance_ratio_ * 100,
                  "chance_level_%": chance},
                 index=[f"PC{i + 1}" for i in range(len(chance))]).to_csv(output_dir / "explained_variance.csv")
    print(f"Saved to {output_dir}")

### Reporting checklist

For a figure legend or a methods section, give:

* **The features:** the list of features, which ones were log-transformed, that they were z-scored, and how missing values were handled (with the % of cells kept per condition).
* **The components:** the variance explained by PC1 and PC2, how many components were above chance (parallel analysis), and what each PC represents, from the loadings.
* **The plot:** *"Cells (small dots), condition centroids (large dots) with 95 % confidence ellipses from a hierarchical bootstrap over experiments and cells; diamonds are experiment centroids."*
* **The test:** *"PERMANOVA on z-scored features (Euclidean, 9999 permutations) at the level of experiment centroids"*, with R², and PERMDISP for dispersion. With 3 replicates, also the PC-score tests (Welch's *t*-test on experiment medians, BH-corrected).
* **The expression check:** whether the PCs correlate with expression level.
* **The exclusions:** the number of cells excluded and the reason.

Example sentence:
> *PCA of 12 foci features (PC1 41 %, PC2 18 % of variance; 2 components above chance) separated MGS3 from WT along PC1, which reflects the amount of phase separation (foci number, area fraction and signal in foci; r > 0.8). The shift was consistent across experiments (PC1 score, Welch's t-test on experiment medians, q = 0.01, n = 4) and not explained by expression level (Spearman ρ of PC1 with nuclear intensity = 0.12).*